# Objective O3 Exploration: Learning Path Course Recommendations

This notebook explores the course catalog datasets and develops the matching strategy for **Objective O3**:
> Given missing skills in a career learning path (e.g., *GST → TDS → bank reconciliation*), recommend up to ~3 relevant online courses, prioritizing free Indian government courses (**NPTEL / SWAYAM**) first, falling back to **Coursera**.

### Dataset Overview
1. **NPTEL / SWAYAM Catalog** (`data/raw/courses/nptel_swayam_courses.csv`):
   - 21,761 courses (SWAYAM and NPTEL) with title, provider, domain, summary, and URL.
   - Free Indian government courses from IITs, IISc, IIMs, and central universities.
   - Contains repeated offerings across semester runs (~58% duplicate titles) requiring canonical deduplication.
2. **Coursera Course Metadata 2025** (`data/raw/courses/coursera_courses_2025.csv`):
   - 5,411 English courses with structured, comma-separated `skills` tags.
   - CC BY-NC-SA 4.0 license.
   - Normalized with `normalise_skill()` and `configs/skill_aliases.yaml`.

### Sections
| Section | Topic |
|---|---|
| **1** | Environment setup & data loading |
| **2** | NPTEL / SWAYAM deduplication analysis |
| **3** | Extracting learning-path core skills from `models/skill_profiles.json` |
| **4** | Coursera skill tag normalization & indexing |
| **5** | NPTEL text search: phrase matching & TF-IDF similarity |
| **6** | Edge case handling: vague skills & boundary-safe regex (C, C++, R, .NET) |
| **7** | Course mapping pipeline execution |
| **8** | Coverage & relevance evaluation |
| **9** | Real learning path recommendations for sample roles |

In [1]:
import os, re, json, collections
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # Run from repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.utils import CONFIGS, MODELS, RAW, REPORTS, normalise_skill, load_aliases, load_params
from src.models.courses import deduplicate_nptel, parse_coursera_courses, make_skill_pattern, map_skills_to_courses

## 1. Load Raw Datasets
Let's inspect the shapes, columns, and sample entries of both raw catalogs.

In [2]:
df_nptel_raw = pd.read_csv(RAW / "courses" / "nptel_swayam_courses.csv")
df_coursera_raw = pd.read_csv(RAW / "courses" / "coursera_courses_2025.csv")

print(f"NPTEL/SWAYAM raw shape: {df_nptel_raw.shape}")
print(f"Coursera raw shape:      {df_coursera_raw.shape}")

print("\n--- NPTEL Sample ---")
display(df_nptel_raw[['title', 'provider', 'organisation', 'url']].head(3))

print("\n--- Coursera Sample ---")
display(df_coursera_raw[['name', 'category', 'skills', 'url']].head(3))

## 2. Deduplicating NPTEL / SWAYAM Offerings

About 58% of the raw rows in `nptel_swayam_courses.csv` represent repeats of the same course run across consecutive semesters or platforms (many starting with `NOC:` prefixes).
We deduplicate by canonicalizing titles (stripping `NOC:`, lowercase, punctuation normalization) and selecting the run with the longest summary and valid URL.

In [3]:
df_nptel = deduplicate_nptel(df_nptel_raw)

print(f"Raw NPTEL/SWAYAM rows:        {len(df_nptel_raw)}")
print(f"Unique canonical courses:     {len(df_nptel)}")
print(f"Duplicate courses collapsed:  {len(df_nptel_raw) - len(df_nptel)} ({(len(df_nptel_raw) - len(df_nptel)) / len(df_nptel_raw):.1%})")

print("\nProvider distribution after deduplication:")
print(df_nptel['provider'].value_counts())

## 3. Extracting Learning-Path Core Skills

The learning path recommends skills that appear as `core_skills` in our trained role profiles (`models/skill_profiles.json`).
These represent the ~750 essential skills across all 103 target job roles.

In [4]:
with open(MODELS / "skill_profiles.json", encoding="utf-8") as f:
    profiles = json.load(f)

roles = profiles["roles"]
core_skills_counter = collections.Counter()
for r, data in roles.items():
    for s in data.get("core_skills", {}):
        core_skills_counter[s] += 1

core_skills = sorted(list(core_skills_counter.keys()))
print(f"Total canonical roles: {len(roles)}")
print(f"Total unique core learning-path skills: {len(core_skills)}")

print("\nTop 15 most frequent core skills across roles:")
for s, count in core_skills_counter.most_common(15):
    print(f"  {s:<25} (in {count} roles)")

## 4. Coursera Skill Tag Normalization

Coursera courses provide a structured `skills` column. We normalize each tag with `normalise_skill()` and resolve aliases via `configs/skill_aliases.yaml`.

In [5]:
aliases = load_aliases(CONFIGS / "skill_aliases.yaml")
coursera_by_skill = parse_coursera_courses(df_coursera_raw, aliases)

print(f"Total distinct normalized Coursera skill tags: {len(coursera_by_skill)}")
overlap = [s for s in core_skills if s in coursera_by_skill]
print(f"Direct exact-tag overlap with core skills: {len(overlap)} / {len(core_skills)} ({len(overlap)/len(core_skills):.1%})")

print("\nSample Coursera matches:")
for s in ['python', 'react.js', 'aws', 'financial accounting', 'machine learning']:
    if s in coursera_by_skill:
        print(f"  [{s}] -> {coursera_by_skill[s][0]['title']}")

## 5. Text Matching on NPTEL / SWAYAM

NPTEL and SWAYAM have no pre-extracted skill tags, but have rich course titles and course summaries.
We use a two-tiered search strategy:
1. **Exact phrase match in title** using boundary-safe regex (handles symbols like `c++`, `c#`, `.net`).
2. **TF-IDF cosine similarity** over title (weighted 3x) + summary with multi-word term coverage filters.

In [6]:
docs = df_nptel['clean_title'] + ' . ' + df_nptel['clean_title'] + ' . ' + df_nptel['clean_title'] + ' . ' + df_nptel['clean_summary']
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
X_nptel = tfidf.fit_transform(docs)

test_queries = ['gst', 'accounting', 'data structures', 'python', 'digital marketing']
Q = tfidf.transform(test_queries)
sims = cosine_similarity(Q, X_nptel)

for i, q in enumerate(test_queries):
    top_idx = sims[i].argsort()[::-1][0]
    best_course = df_nptel.iloc[top_idx]['clean_title']
    score = sims[i][top_idx]
    print(f"Query: {q:<18} -> [{score:.3f}] {best_course}")

## 6. Edge Case Handling: Vague Skills & Boundary Safety

1. **Vague vocabulary skills:** Words like `core`, `development`, `management`, `hospital`, `general` are too broad in isolation. Returning a course on 'Hospital Waste Management' for 'hospital' is noisy junk. We map them to an empty list `[]`.
2. **Short skill names:** For single letters like `c`, boundary checks ensure we match `Programming in C` while rejecting `C++`, `C#`, and historical date abbreviations like `(c. 1900)`.
3. **Polysemy disambiguation:** Disambiguates `angular` (web framework vs. angular momentum in aerospace).

In [7]:
p_c = make_skill_pattern('c')
print("Boundary safety for 'c':")
print("  'Programming in C' matches:   ", bool(p_c.search('Programming in C')))
print("  'Programming in C++' matches: ", bool(p_c.search('Programming in C++')))
print("  'C# Development' matches:     ", bool(p_c.search('C# Development')))
print("  'Modern Art (c. 1920)' matches:", bool(p_c.search('Modern Art (c. 1920)')))

p_cpp = make_skill_pattern('c++')
print("\nBoundary safety for 'c++':")
print("  'Programming in C++' matches: ", bool(p_cpp.search('Programming in C++')))
print("  'Programming in C' matches:   ", bool(p_cpp.search('Programming in C')))

## 7. Quality Metrics & Relevance

Let's load the generated metrics report (`reports/courses_summary.json`) and sample relevance report (`reports/courses_relevance_sample.csv`).

In [8]:
with open(REPORTS / "courses_summary.json", encoding="utf-8") as f:
    summary = json.load(f)

print("=== Courses Quality Metrics ===")
for k, v in summary.items():
    print(f"  {k:<30}: {v}")

df_sample = pd.read_csv(REPORTS / "courses_relevance_sample.csv")
print(f"\nSample relevance check ({len(df_sample)} pairs):")
display(df_sample.head(10))

## 8. Learning Path Recommendations for Real Job Roles

Let's simulate what a user sees when exploring their recommended learning path.

In [9]:
with open(MODELS / "skill_courses.json", encoding="utf-8") as f:
    skill_courses = json.load(f)

sample_roles = ["Accountant", "Frontend Developer", "Data Scientist", "Mechanical Engineer"]

for role in sample_roles:
    if role not in roles:
        continue
    print(f"\n{'='*70}")
    print(f"Role: {role} ({roles[role]['family']})")
    print(f"{'='*70}")
    core = list(roles[role]['core_skills'].keys())[:4]
    for s in core:
        courses = skill_courses.get(s, [])
        print(f"\n  Skill: [{s.upper()}]")
        if not courses:
            print("    ↳ (no course found - empty list fallback)")
        else:
            for c in courses:
                print(f"    ↳ [{c['provider']}] {c['title']}")
                print(f"      URL: {c['url']}")